In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import chess

device = torch.device("cuda")
print("Using:", device)

Using: cuda


In [2]:
FEATURES = 64 * 12 * 64
HIDDEN = 128   # 🔥 nagyobb = jobb NNUE

In [3]:
def nnue_index(king_sq, piece_sq, piece_type, color):
    return (
        king_sq * 12 * 64 +
        (piece_type - 1 + (0 if color else 6)) * 64 +
        piece_sq
    )

In [4]:
def board_to_tensor(board):

    x = torch.zeros(FEATURES, device=device)

    wk = board.king(chess.WHITE)
    bk = board.king(chess.BLACK)

    for sq, piece in board.piece_map().items():

        idx_w = nnue_index(wk, sq, piece.piece_type, piece.color)
        idx_b = nnue_index(bk, sq, piece.piece_type, piece.color)

        x[idx_w] += 1.0
        x[idx_b] -= 1.0

    return x

In [5]:
class NNUEDataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.data = []

        with open(path) as f:
            for line in f:
                fen, score = line.split("|")
                score = int(score)

                # clamp
                score = max(-1000, min(1000, score))

                self.data.append((fen.strip(), score))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fen, score = self.data[idx]

        board = chess.Board(fen)
        x = board_to_tensor(board)

        y = score / 400.0

        return x, y
    

In [6]:
dataset = NNUEDataset("train.txt")

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2048,
    shuffle=True,
    num_workers=0
)

In [7]:
class NNUE(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(FEATURES, HIDDEN)
        self.fc2 = nn.Linear(HIDDEN, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x.squeeze(-1)

In [8]:
import time

model = NNUE().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(5):

    start_time = time.time()
    total_loss = 0

    for i, (x, y) in enumerate(loader):

        batch_start = time.time()

        x = x.to(device)
        y = y.to(device).float()

        preds = model(x)
        loss = loss_fn(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ===== PROGRESS PRINT =====
        if i % 10 == 0:

            elapsed = time.time() - start_time
            batches_done = i + 1
            total_batches = len(loader)

            samples_done = batches_done * x.size(0)
            speed = samples_done / elapsed

            eta = (total_batches - batches_done) * (elapsed / batches_done)

            # GPU memória
            if torch.cuda.is_available():
                mem = torch.cuda.memory_allocated() / 1024**2
            else:
                mem = 0

            print(
                f"[Epoch {epoch}] "
                f"Batch {i}/{total_batches} | "
                f"Loss: {loss.item():.4f} | "
                f"Avg: {total_loss / batches_done:.4f} | "
                f"{speed:.0f} samples/s | "
                f"ETA: {eta:.1f}s | "
                f"VRAM: {mem:.0f}MB"
            )

    epoch_time = time.time() - start_time

    print(
        f"\n🔥 Epoch {epoch} DONE | "
        f"Avg Loss: {total_loss / len(loader):.4f} | "
        f"Time: {epoch_time:.1f}s\n"
    )

[Epoch 0] Batch 0/9 | Loss: 0.4460 | Avg: 0.4460 | 295 samples/s | ETA: 55.5s | VRAM: 496MB

🔥 Epoch 0 DONE | Avg Loss: 0.3801 | Time: 53.1s

[Epoch 1] Batch 0/9 | Loss: 0.3376 | Avg: 0.3376 | 337 samples/s | ETA: 48.6s | VRAM: 496MB

🔥 Epoch 1 DONE | Avg Loss: 0.3368 | Time: 51.5s

[Epoch 2] Batch 0/9 | Loss: 0.2929 | Avg: 0.2929 | 367 samples/s | ETA: 44.7s | VRAM: 496MB

🔥 Epoch 2 DONE | Avg Loss: 0.3088 | Time: 49.7s

[Epoch 3] Batch 0/9 | Loss: 0.3010 | Avg: 0.3010 | 352 samples/s | ETA: 46.5s | VRAM: 496MB

🔥 Epoch 3 DONE | Avg Loss: 0.2882 | Time: 51.7s

[Epoch 4] Batch 0/9 | Loss: 0.2699 | Avg: 0.2699 | 342 samples/s | ETA: 48.0s | VRAM: 496MB

🔥 Epoch 4 DONE | Avg Loss: 0.2712 | Time: 53.9s



In [11]:
fc1 = model.fc1
fc2 = model.fc2

weights = {
    "W1": fc1.weight.detach().cpu().numpy().T.astype(np.float32),
    "B1": fc1.bias.detach().cpu().numpy().astype(np.float32),
    "W2": fc2.weight.detach().cpu().numpy().flatten().astype(np.float32),
    "B2": float(fc2.bias.detach().cpu().item())
}

np.savez("nnue_weights.npz", **weights)
print("Saved!")

Saved!


In [12]:
def load(self, path):
    data = np.load(path)

    self.W1 = data["W1"]
    self.B1 = data["B1"]
    self.W2 = data["W2"]
    self.B2 = float(data["B2"])